Silver Summary Table QA

# Daily KPI Summary Validation Notebook

This notebook validates the daily KPI summary table used in the Topline dashboard.  
It compares key metrics from the `SILVER_EVENT_DIM` table with those in the summary table to ensure consistency and accuracy.

## Items Checked

### Daily KPI

- **A1. Overall Metrics**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A2. Metrics by Day**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A3. Metrics by Source**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A4. Metrics by Country**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A5. Metrics by User Type**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A6. Metrics by Edition**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A7. Metrics by Simplified Referrer**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A8. Metrics by Region**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %

- **B1. Metrics by Source and Referrer**

- **C1. Metrics by Day and Source**  
- **C2. Metrics by Day and User Type**  
- **C3. Metrics by User Type and Source**  
- **C4. Metrics by User Type and Referrer**  
- **C5. Metrics by User Type and Country**

- **D1. Metrics by Day, Source, Referrer, Edition**  
- **D2. Metrics by Day, Source, User Type, Referrer, Country**

- **E1. Daily Trend for Date Range Jan 4–6**  
- **E2. Sessions by Referrer Pie Chart**

### Content

- **A1. Overall**  
- **A2. By Day**  
- **A3. By Topic Channel**  
- **A4. By User Type**  
- **A5. By Country**  
- **A6. By Region**  
- **A7. By Subschannel**  
- **A8. By Referrer**  
- **A9. By Edition**

- **B1. Users and PV of Top 10 Canonical URLs (Jan 5–9)**  
- **B2. Users and PV of Top 10 Topic Channels (Jan 5–9)**  
- **B3. Users and PV of Top 10 Landing Pages (Jan 5–9)**  
- **B4. Users and PV of Top 10 Articles (Jan 5–9)**

---
### Review on Dec 9, 2025
- Review after the update on the logic: user count logic is changed to coalesce(user_dim_id, anonymous_user_id) and bot filter is applied
- Daily KPI: items 1-7 & D1, D2 were passed.
- Monthly KPI: items 1-7, 11, 14 were passed.
- Daily content: items A1–9 and B1–4 were passed.
- Monthly KPI: items 1-8 were passed.

### Review on Nov 4, 2025
- Jan 1–31, 2025 data was used for review.
- The numbers in Tableau (`TOPLINE_SILVER_V1` file and the summary tables) are aligned with the figures pulled from `SILVER_EVENT_DIM`.
- One exception is the **session count**. Theoretically, `USER_DIM_ID` should not be null when `application_authenticated_state` is either "registered" or "subscriber", and the query is built based on that assumption. However, in rare cases, `USER_DIM_ID` is recorded as null even when `application_authenticated_state` is "registered" or "subscriber".
- Due to this technical glitch, the number of sessions by user type may differ slightly for "subscriber" and "registered" users. The discrepancy is less than 1%, and the review considers the numbers aligned if the difference in session count is below 1% for these user types.  

In [1]:
# Import libraries 
import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
import sys

In [2]:
# Set up the data connection 
load_dotenv(dotenv_path='jk.env')
con = snowflake.connector.connect(
    user="joonkyung.kim@thomsonreuters.com", #You can get it by executing in UI: desc user <username>;
    account="THOMSONREUTERS-A206448_PROD", #Add all of the account-name between https:// and snowflakecomputing.com
    authenticator="externalbrowser",
)
cur = con.cursor()

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.thomsonreuters.com/idp/SSO.saml2?SAMLRequest=nZLBjtowEIZfJXLPsZ0shWABKwpaLSpbKAkt6qUyiQELx049DoG3rwmLtD3sHnqLnG%2FG3%2FifweO5VMFJWJBGD1GEKQqEzk0h9X6I1tlTmKAAHNcFV0aLIboIQI%2BjAfBSVWxcu4NeiT%2B1ABf4RhpY%2B2OIaquZ4SCBaV4KYC5n6fhlzmJMWWWNM7lR6E3JxxUcQFjnDe8lBUivd3CuYoQ0TYObB2zsnsSUUkL7xFNX5NOdP%2FuZ3uEjQjtX3hMeX766fZH69gQfaW1vELDnLFuGy0WaoWB8V50YDXUpbCrsSeZivZrfBMAbjGPa7XSS38vVYoprCAUHF0YYtGl2ih9Fbsqqdr419l9kJwqizF766WfTIaqOsnBntXGH2ap%2FFJvtz%2B%2F7hm%2BO69U2%2FXqyurwktezNn3s2ellPIEfBj3u88TXeGUAtZvoaqvNHNO6GtBPSJIse2Oc%2B63RxP%2Bn9QsHUhyo1d23l3RzAYHcwJRhtRe1841ZRFhVJ0wW%2BRhmj23qw9iI7%2Bq%2BhB%2BRti9d1%2B%2BYTmE2XRsn8EjwZW3L3fkARjtoTWYS7FmWi5FKNi8IKAB%2BUUqaZWMGd32pna4HI6Hbrv3s9%2Bgs%3D&RelayState=ver%3A3-hint%3A7475593065661118-ETMsDgAAAZ1tZFwrABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEKtMuwHlCaESi0XdB5n5Q5UAAACw6DPRrbVNxT6LWBo8Id1Tgv7P%2Fm463cioXrvwkg51duSI7lfPBYoFPxtj9Gni%

In [8]:
start_date = '2026-01-01'
end_date = '2026-01-01'
source_filter = "AND SOURCE IN ('desktop', 'mobileweb')"
date_set_a_start = '2025-12-15'
date_set_a_end = '2025-12-31'
date_set_b_start = '2025-12-15'
date_set_b_end = '2025-12-31'
date_range = "('2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-09', '2025-01-10')" #,'2025-02-02', '2025-03-01')"

In [4]:
# cur.execute("""USE ROLE A208946_REUTERS_ANALYTICS_MDS_READ_WRITE""")
cur.execute("""USE ROLE REUTERS_CDP_QA_ANALYTICS""")
cur.execute("""USE WAREHOUSE A208946_REUTERS_ANALYTICS_MDS_WH""")

## Functions

In [5]:
import numpy as np

def compare_dataframes(df1, df2, column_list, content_flag):
    # Step 1: Sort both dataframes
    sorted_df1 = df1.sort_values(by=column_list).reset_index(drop=True)
    sorted_df2 = df2.sort_values(by=column_list).reset_index(drop=True)

    if content_flag:
        comparison_result = sorted_df1.fillna(np.nan).equals(sorted_df2.fillna(np.nan))
        if comparison_result:
            return 'PASS'
        else:
            return 'FAIL'
    
    else:
        # Step 2: Handle null-safe comparison
        comparison_result = sorted_df1.fillna(np.nan).equals(sorted_df2.fillna(np.nan))

        # Step 3: Group by column_list and sum 'SESSIONS'
        df1_grouped_sessions = df1.groupby(column_list)['SESSIONS'].sum()
        df2_grouped_sessions = df2.groupby(column_list)['SESSIONS'].sum()

        # Step 4: Create comparison DataFrame
        comparison = pd.DataFrame({
            'SESSIONS_df1': df1_grouped_sessions,
            'SESSIONS_df2': df2_grouped_sessions
        })

        # Step 5: Calculate percentage difference
        comparison['%_DIFFERENCE'] = ((comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']) / comparison['SESSIONS_df1']) * 100
        print("\nSession comparison:\n", comparison)

        # Step 6: Drop 'SESSIONS' column for final comparison
        df1_excluded = sorted_df1.drop(columns=['SESSIONS'])
        df2_excluded = sorted_df2.drop(columns=['SESSIONS'])

        # Step 7: Null-safe comparison for excluded DataFrames
        null_safe_equal = df1_excluded.fillna(np.nan).equals(df2_excluded.fillna(np.nan))

        # Step 8: Final pass/fail check
        if (comparison['%_DIFFERENCE'].abs() < 1).all() and null_safe_equal:
            print('PASS - difference in sessions is less than 1% and dataframes match')
        else:
            print('FAIL - either session difference exceeds 1% or dataframes do not match')

def normalize(val):
    return '' if pd.isna(val) or val == '' else str(val)

def compare_monthly_dataframes(df1, df2, column_list, threshold):
    null_safe_equal = False
    # Step 1: Sort both dataframes
    placeholder = ''
    df1_safe = df1.fillna(placeholder)
    df2_safe = df2.fillna(placeholder)

    sorted_df1 = df1_safe.sort_values(by=column_list).reset_index(drop=True)
    sorted_df2 = df2_safe.sort_values(by=column_list).reset_index(drop=True)

    # Step 6: Drop 'SESSIONS' column for final comparison
    df1_excluded = sorted_df1.drop(columns=['SESSIONS'])
    df2_excluded = sorted_df2.drop(columns=['SESSIONS'])

    # Step 7: Null-safe comparison for excluded DataFrames
    null_safe_equal_excl_sessions = (df1_excluded.astype(str) == df2_excluded.astype(str)).all().all()

    # Step 3: Group by column_list and sum 'SESSIONS'
    df1_grouped_sessions = sorted_df1.groupby(column_list)['SESSIONS'].sum()
    df2_grouped_sessions = sorted_df2.groupby(column_list)['SESSIONS'].sum()

    # Step 4: Create comparison DataFrame
    comparison = pd.DataFrame({
        'SESSIONS_df1': df1_grouped_sessions,
        'SESSIONS_df2': df2_grouped_sessions
    })

    # Step 5: Calculate percentage difference
    comparison['SESSIONS_df1'] = comparison['SESSIONS_df1'].astype(float)
    comparison['SESSIONS_df2'] = comparison['SESSIONS_df2'].astype(float)

        
    comparison['DIFF'] = (comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']).abs()
    comparison['%_DIFFERENCE'] = ((comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']) / comparison['SESSIONS_df1']) * 100
    print("\nSession comparison:\n", comparison)

    if ((comparison['DIFF'] < threshold).all() or (comparison['%_DIFFERENCE'].abs() < 1).all()) and null_safe_equal_excl_sessions:
        null_safe_equal = True

    #null_safe_equal = (sorted_df1.astype(str) == sorted_df2.astype(str)).all().all()

    # Step 8: Final pass/fail check
    if null_safe_equal:
        return 'PASS'
    else:
        return 'FAIL'


def compare_monthly_content(df1, df2, column_list, threshold):
    null_safe_equal = False
    # Step 1: Sort both dataframes
    #placeholder = ''
    #df1_safe = df1.fillna(placeholder)
    #df2_safe = df2.fillna(placeholder)
    
    df1_safe = df1.apply(lambda col: col.map(normalize))
    df2_safe = df2.apply(lambda col: col.map(normalize))

    sorted_df1 = df1_safe.sort_values(by=column_list).reset_index(drop=True)
    sorted_df2 = df2_safe.sort_values(by=column_list).reset_index(drop=True)
    null_safe_equal = (sorted_df1.astype(str) == sorted_df2.astype(str)).all().all()
    
    """
    if 'USER_TYPE' in column_list:
        # Step 6: Drop 'SESSIONS' column for final comparison
        df1_excluded = sorted_df1.drop(columns=['SESSIONS'])
        df2_excluded = sorted_df2.drop(columns=['SESSIONS'])

        # Step 7: Null-safe comparison for excluded DataFrames
        null_safe_equal_excl_sessions = (df1_excluded.astype(str) == df2_excluded.astype(str)).all().all()

        # Step 3: Group by column_list and sum 'SESSIONS'
        df1_grouped_sessions = sorted_df1.groupby(column_list)['SESSIONS'].sum()
        df2_grouped_sessions = sorted_df2.groupby(column_list)['SESSIONS'].sum()

        # Step 4: Create comparison DataFrame
        comparison = pd.DataFrame({
            'SESSIONS_df1': df1_grouped_sessions,
            'SESSIONS_df2': df2_grouped_sessions
        })

        # Step 5: Calculate percentage difference
        comparison['SESSIONS_df1'] = comparison['SESSIONS_df1'].astype(float)
        comparison['SESSIONS_df2'] = comparison['SESSIONS_df2'].astype(float)

        
        comparison['DIFF'] = (comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']).abs()
        comparison['%_DIFFERENCE'] = ((comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']) / comparison['SESSIONS_df1']) * 100
        print("\nSession comparison:\n", comparison)

        if ((comparison['DIFF'] < threshold).all() or (comparison['%_DIFFERENCE'].abs() < 1).all()) and null_safe_equal_excl_sessions:
            null_safe_equal = True

    else:
        null_safe_equal = (sorted_df1.astype(str) == sorted_df2.astype(str)).all().all()
    """
    # Step 8: Final pass/fail check
    if null_safe_equal:
        return 'PASS'
    else:
        return 'FAIL'
    
def run_query(query):
    cur.execute(query)
    rows = cur.fetchall()
    cols = [col[0] for col in cur.description]
    df = pd.DataFrame(rows, columns=cols)
    #df = cur.fetch_pandas_all()

    metric_cols = [
        'USER_COUNT', 'SESSIONS', 'PAGEVIEWS',
        'VIDEO_STORY_START', 'VIDEO_STORY_FINISHED',
        'ARTICLE_RECIRCULATION', 'ARTICLE_BOTTOM_REACH'
    ]

    # Only process columns that exist in the dataframe
    for col in (c for c in metric_cols if c in df.columns):
        if df[col].dtype == 'object':
            df[col] = df[col].astype('int64')
    return df
    
def derived_metrics(df1):
    df1['PV/U'] = df1['PV'] / df1['USER_COUNT']
    df1['PV/S'] = df1['PV'] / df1['SESSIONS']
    df1['sessions/U'] = df1['SESSIONS'] / df1['USER_COUNT']
    df1['bottom_reach_perc'] = df1['ARTICLE_BOTTOM_REACH'] / df1['PV']
    df1['recirculation_perc'] = df1['ARTICLE_RECIRCULATION'] / df1['PV']
    df1['video_completion_perc'] = df1['VIDEO_STORY_FINISHED'] / df1['VIDEO_STORY_START']
    
    return df1


## Query for Daily KPI

In [15]:
q1_overall = f"""select
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count, --count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' 
    and segment_user_agent_type='other user agent' {source_filter}"""

q1s_overall = f"""select
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' {source_filter}"""

q2_daily = f"""select d,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count, --count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
where d between '{start_date}' and '{end_date}' and segment_user_agent_type='other user agent' {source_filter} group by 1"""

q2s_daily = f"""select d,
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' {source_filter} group by 1"""

q3_by_source = f"""select source, 
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count, --count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e 
where d between '{start_date}' and '{end_date}' and segment_user_agent_type='other user agent' group by 1"""

q3s_by_source = f"""select source,
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1"""

q4_by_country = f"""select c.country_name as FIRST_COUNTRY,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.ss_SESSION_ID = S.ss_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' group by 1"""

q4s_by_country = f"""select FIRST_COUNTRY,
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1"""

q5_by_user_type = f"""select
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_session_id) as sessions,
    count(distinct iff(e.event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' group by 1"""

q5s_by_user_type = f"""select USER_TYPE,
    count(distinct UID_BY_USERTYPE) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1"""

q6_by_edition = f"""select
    ED.EDITION as first_edition,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' group by 1"""

q6s_by_edition = f"""select first_edition,
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1"""

q7_by_referrer = f"""select
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' group by 1"""

q7s_by_referrer = f"""select
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}'  group by 1"""

q8_by_region = f"""select
    c.region as first_region,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}'  
    and e.segment_user_agent_type='other user agent' group by 1"""

q8s_by_region = f"""select
    FIRST_REGION,
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}'  group by 1"""

In [7]:
b1_by_source_referrer = f"""select
    S.source,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' group by 1,2"""

b1s_by_source_referrer = f"""select
    source,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct ss_anonymous_user_id) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2"""

c1_by_day_source = f"""select
    e.d, s.source,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' group by 1,2"""

c1s_by_day_source = f"""select d,
    source,
    count(distinct ss_anonymous_user_id) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND DAY
c2_by_day_usertype = f"""select e.d,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2"""

c2s_by_day_usertype = f"""select d,
    USER_TYPE,
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND SOURCE
c3_by_source_usertype = f"""select 
    s.source,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2"""

c3s_by_source_usertype = f"""select 
    source,
    USER_TYPE,
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND REFERRER
c4_by_usertype_referrer = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON E.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2"""

c4s_by_usertype_referrer = f"""select 
    USER_TYPE,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND COUNTRY
c5_by_usertype_country = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    C.COUNTRY_NAME AS FIRST_COUNTRY,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2"""

c5s_by_usertype_country = f"""select 
    USER_TYPE,
    FIRST_COUNTRY,
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2"""

d1_by_day_source_referrer_edition = f"""select e.d,
    s.source,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    ED.EDITION AS FIRST_EDITION,
    count(distinct COALESCE(E.USER_DIM_ID, E.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' group by 1,2,3,4"""

d1s_by_day_source_referrer_edition = f"""select d,
    source,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    FIRST_EDITION,
    count(distinct COALESCE_UID) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2,3,4"""

# BY D, USER TYPE, SOURCE, REFERRER
d2_by_day_source_referrer_region = f"""select e.d,
    s.source,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    C.REGION AS FIRST_REGION,
    count(distinct COALESCE(E.USER_DIM_ID, E.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' group by 1,2,3,4"""

d2s_by_day_source_referrer_region = f"""select d,
    source,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    FIRST_REGION,
    count(distinct COALESCE_UID) as user_count,
    sum(SESSIONS_for_usertype_all) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' group by 1,2,3,4"""

# To review the % change
e1 = f"""select 
    iff(e.d between '{date_set_a_start}' and '{date_set_a_end}', 'set_a', iff(e.d between '{date_set_b_start}' and '{date_set_b_end}', 'set_b', null)) date_set,
    s.source,
    C.region_name AS FIRST_REGION,
    count(distinct E.SS_ANONYMOUS_USER_ID) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{date_set_a_start}' and '{date_set_b_end}' group by 1,2,3"""

e1s = f"""select
    iff(d between '{date_set_a_start}' and '{date_set_a_end}', 'set_a', iff(d between '{date_set_b_start}' and '{date_set_b_end}', 'set_b', null)) date_set,
    source,
    FIRST_REGION,
    count(distinct ss_anonymous_user_id) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{date_set_a_start}' and '{date_set_b_end}' group by 1,2,3"""

## Query for daily content 

In [130]:
# VERSION 2 - Dec 09
# by topic channel, by user type, overall, by day, ETC. 
c_a1_overall = f"""select
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' ;"""

c_a1s_overall = f"""
    with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0)
    select 
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}';"""

c_a2_by_day = f"""select d,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
    group by 1;"""

c_a2s_by_day = f"""
    with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0)
    select d,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

c_a3_by_topic_ch = f"""select topic_channel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' 
    group by 1 having pv > 0;"""

c_a3s_by_topic_ch = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0)
    select topic_channel,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

c_a4_by_usertype = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
        ELSE NULL END AS user_type,
    count(distinct iff(event_type = 'page', IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
    group by 1 HAVING PV > 0;"""

c_a4s_by_usertype = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select user_type,
    count(distinct uid_by_usertype) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

c_a5_by_country_tch = f"""select 
    c.country_name as first_country,
    topic_channel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
    group by 1,2 HAVING PV > 0;"""

c_a5s_by_country_tch = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select first_country, topic_channel,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1,2;"""

# A6. by region
c_a6_by_region = f"""select 
    c.region as first_region,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
    group by 1 HAVING PV > 0;"""

c_a6s_by_region = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0)
    select first_region,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

# A7. by subschannel
c_a7_by_subchannel = f"""select 
    topic_subchannel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
    group by 1 HAVING PV > 0;"""

c_a7s_by_subchannel = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0)
    select topic_subchannel,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""


# A8. by referrer 
c_a8_by_referrer = f"""select 
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
    group by 1 HAVING PV > 0;"""

c_a8s_by_referrer = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select first_utm_referrer_simplified,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

# A9. By edition 
c_a9_by_edition = f"""select 
    ED.EDITION AS FIRST_EDITION,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
    group by 1 HAVING PV > 0;"""

c_a9s_by_edition = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select FIRST_EDITION,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1 
where d between '{start_date}' and '{end_date}' group by 1;"""

# B1. Users and PV of top 10 canonical URL for SET A 
c_b1_top_canonical_url = f"""select CANONICAL_URL,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
        group by 1 ORDER BY PV DESC LIMIT 10;"""
    
c_b1s_top_canonical_url = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select CANONICAL_URL,
    count(distinct COALESCE_UID) as user_count,
    SUM(PAGEVIEWS) AS PV
FROM T1
where d between '{start_date}' and '{end_date}' group by 1 ORDER BY PV DESC LIMIT 10;"""

# B2. Users and PV of top 10 topic channel for Jan 1-5 & desktop
c_b2_top_topic_channel_by_source = f"""select source as first_source, topic_channel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
        and source = 'desktop'
        group by 1,2 ORDER BY PV DESC LIMIT 10;"""
    
c_b2s_top_topic_channel_by_source = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select first_source, topic_channel,
    count(distinct COALESCE_UID) as user_count,
    SUM(PAGEVIEWS) AS PV
FROM T1
where d between '{start_date}' and '{end_date}' and first_source = 'desktop' group by 1,2 ORDER BY PV DESC LIMIT 10;"""

# B3. Users and PV of top 10 landing page for Jan 1-5
c_b3_top_landing_page_user_type = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
        ELSE NULL END AS user_type,
    content_type,
    CANONICAL_URL, 
    count(distinct iff(event_type = 'page', IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
        and user_type = 'subscriber'
        and content_type = 'landing page'
        group by 1,2,3 ORDER BY PV DESC LIMIT 10;"""
    
c_b3s_top_landing_page_user_type = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select user_type, content_type, CANONICAL_URL,
    COUNT(DISTINCT UID_BY_USERTYPE) AS USER_COUNT,
    SUM(PAGEVIEWS) AS PV
    FROM T1
    where d between '{start_date}' and '{end_date}' 
        and user_type = 'subscriber' and content_type = 'landing page' group by 1,2,3 ORDER BY PV DESC LIMIT 10;"""

# B4. Users and PV of top 10 top article for Jan 1-5
c_b4_top_article = f"""select 
    content_type,
    CANONICAL_URL, 
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
        and content_type = 'article'
        group by 1,2 ORDER BY PV DESC LIMIT 10;"""
    
c_b4s_top_article = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_CONTENT_SUMMARY where pageviews > 0) select content_type, CANONICAL_URL,
    count(distinct COALESCE_UID) as user_count,
    SUM(PAGEVIEWS) AS PV
    FROM T1
    where d between '{start_date}' and '{end_date}' 
        and content_type = 'article' group by 1,2 ORDER BY PV DESC LIMIT 10;"""

## Daily KPI Review

In [10]:
cur.execute(q1_overall)
rows = cur.fetchall()
cols = [col[0] for col in cur.description]
df1 = pd.DataFrame(rows, columns=cols)

cur.execute(q1s_overall)
rows = cur.fetchall()
cols = [col[0] for col in cur.description]
df1s = pd.DataFrame(rows, columns=cols)
#df1s = cur.fetch_pandas_all()

In [11]:
df1 = derived_metrics(df1)
df1s = derived_metrics(df1s)

In [12]:
if (df1 == df1s).all().all():
    result = "Daily KPI - A1: Pass"
else:
    result = "Fail"
print(result)
df1 == df1s

Fail


,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,True,False,True,True,True,True,True,True,False,False,True,True,True


In [13]:
df1 # aligned

,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,1391077,1959332,1812390,765921,56547,108880,531868,1.302868,0.925004,1.4085,0.293462,0.060075,0.073829


In [14]:
df1s

,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,1391077,1959344,1812390,765921,56547,108880,531868,1.302868,0.924998,1.408509,0.293462,0.060075,0.073829


In [ ]:
q1_session_debug = f"""
WITH SE AS (select
    DISTINCT E.SS_SESSION_ID
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where d between '{start_date}' and '{end_date}' 
        and segment_user_agent_type='other user agent' {source_filter}),
    SS AS (SELECT DISTINCT SS_SESSION_ID FROM FROM mydataspace.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY S
    where d between '{start_date}' and '{end_date}')
    SELECT  
    
    
    """

q1s_overall = f"""select
    count(distinct coalesce_uid) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '{start_date}' and '{end_date}' {source_filter}"""

In [16]:
df2 = run_query(q2_daily)
df2s = run_query(q2s_daily)

2026-01-01	1391078	1959332	1812390
D	COUNT(DISTINCT COALESCE_UID)	SUM(SESSIONS_FOR_USERTYPE_ALL)	SUM(PAGEVIEWS)
2026-01-01	1391077	1959344	1812390

In [17]:
df2

,D,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,2026-01-01,1391077,1959332,1812390,765921,56547,108880,531868


In [18]:
df2s

,D,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,2026-01-01,1391077,1959344,1812390,765921,56547,108880,531868


In [175]:
df3 = run_query(q3_by_source)
df3s = run_query(q3s_by_source)

In [182]:
df4 = run_query(q4_by_country)
df4s = run_query(q4s_by_country)

In [176]:
df5 = run_query(q5_by_user_type)
df5s = run_query(q5s_by_user_type)

In [177]:
df6 = run_query(q6_by_edition)
df6s = run_query(q6s_by_edition)

In [178]:
df7 = run_query(q7_by_referrer)
df7s = run_query(q7s_by_referrer)

In [179]:
df8 = run_query(q8_by_region)
df8s = run_query(q8s_by_region)

In [19]:
compare_dataframes(df2, df2s, ['D'], False)


Session comparison:
             SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
D                                                   
2026-01-01       1959332       1959344     -0.000612
PASS - difference in sessions is less than 1% and dataframes match


In [184]:
compare_dataframes(df3, df3s, ['SOURCE'], False)


Session comparison:
            SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
SOURCE                                             
desktop         1834890       1834901     -0.000599
mobileweb       2349410       2349412     -0.000085
PASS - difference in sessions is less than 1% and dataframes match


In [185]:
compare_dataframes(df5, df5s, ['USER_TYPE'], False)


Session comparison:
             SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
USER_TYPE                                           
anonymous        3986833       3986866     -0.000828
registered        120279        120277      0.001663
subscriber         86713         86686      0.031137
PASS - difference in sessions is less than 1% and dataframes match


In [186]:
compare_dataframes(df6, df6s, ['FIRST_EDITION'], False)


Session comparison:
                SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
FIRST_EDITION                                          
jp                   411128        411128      0.000000
us                  3768317       3768330     -0.000345
PASS - difference in sessions is less than 1% and dataframes match


In [187]:
compare_dataframes(df7, df7s, ['FIRST_UTM_REFERRER_SIMPLIFIED'], False)


Session comparison:
                                SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
FIRST_UTM_REFERRER_SIMPLIFIED                                          
AI agents                            123496        123496      0.000000
Direct                              1063578       1063585     -0.000658
Email                                 24361         24361      0.000000
Newsletter                            34074         34074      0.000000
Organic                             1992152       1992158     -0.000301
Other                                754229        754229      0.000000
Social                               192410        192410      0.000000
PASS - difference in sessions is less than 1% and dataframes match


In [188]:
compare_dataframes(df8, df8s, ['FIRST_REGION'], False)


Session comparison:
               SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
FIRST_REGION                                          
APAC                316559        316559      0.000000
EMEA                634997        634997      0.000000
LATAM                48214         48214      0.000000
NA                 2819101       2819114     -0.000461
PASS - difference in sessions is less than 1% and dataframes match


In [189]:
compare_dataframes(df4, df4s, ['FIRST_COUNTRY'], False)


Session comparison:
                                     SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
FIRST_COUNTRY                                                               
Afghanistan                                   16            16           0.0
Aland Islands                                  3             3           0.0
Albania                                       38            38           0.0
Algeria                                      116           116           0.0
American Samoa                                43            43           0.0
...                                          ...           ...           ...
Venezuela (Bolivarian Republic of)           351           351           0.0
Viet Nam                                    4376          4376           0.0
Yemen                                         76            76           0.0
Zambia                                        69            69           0.0
Zimbabwe                                      75      

In [ ]:
# E2.Sessions by referrer pie chart PASS: 2025-11-03, 2025-11-09
# Calculate total sum of sessions
total_sessions = df7['SESSIONS'].sum()

# Calculate percentage for each value
df7_pie = df7.copy()
df7_pie['SESSIONS_PERCENT'] = (df7_pie['SESSIONS'] / total_sessions) * 100
df7_pie

,FIRST_UTM_REFERRER_SIMPLIFIED,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,SESSIONS_PERCENT
0,Direct,589519,1063578,1262352,455568,47591,32913,174450,25.418302
1,AI agents,77324,123496,119625,83469,5984,570,36968,2.951414
2,Email,16782,24361,25103,7906,617,1061,3827,0.582200
3,Other,478615,754229,636197,314049,18571,8475,162888,18.025213
4,Organic,1384584,1992152,1809445,1231130,78641,42027,630603,47.610162
5,Newsletter,21233,34074,39449,24772,2380,378,12809,0.814330
6,Social,136618,192410,155918,89639,3559,797,58886,4.598380


In [40]:
dfb1 = run_query(b1_by_source_referrer)
dfb1s = run_query(b1s_by_source_referrer)

In [41]:
compare_dataframes(dfb1, dfb1s, ['SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED'], False)


Session comparison:
                                          SESSIONS_df1  SESSIONS_df2  \
SOURCE    FIRST_UTM_REFERRER_SIMPLIFIED                               
desktop   AI agents                             58291         58291   
          Direct                               452447        452451   
          Email                                 11547         11547   
          Newsletter                            13213         13213   
          Organic                              500958        500961   
          Other                                170651        170651   
          Social                                27293         27293   
mobileweb AI agents                             24051         24051   
          Direct                               258195        258196   
          Email                                  8993          8993   
          Newsletter                             9153          9153   
          Organic                              878946  

In [46]:
cur.execute(c1_by_day_source)
dfc1 = cur.fetch_pandas_all()
cur.execute(c1s_by_day_source)
dfc1s = cur.fetch_pandas_all()

In [ ]:
# C1. by_source_referrer_edition PASS 
sorted_dfc1 = dfc1.sort_values(by=['SOURCE', 'D']).reset_index(drop=True)
sorted_dfc1s = dfc1s.sort_values(by=['SOURCE', 'D']).reset_index(drop=True)
result = "Daily KPI - C1: Pass" if sorted_dfc1.equals(sorted_dfc1s) else "fail"
print(result)

#dfc1.sort_values(by = ['SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_EDITION']).reset_index(drop=True) == dfc1s.sort_values(by = ['SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_EDITION']).reset_index(drop=True)

Daily KPI - C1: Pass


In [ ]:
dfc1.head() #OK

,D,SOURCE,USER_COUNT,SESSIONS,PV
0,2025-01-01,desktop,254059,410860,522867
1,2025-01-08,desktop,641274,1065314,1297352
2,2025-01-02,mobileweb,1176806,1642888,1517018
3,2025-01-08,mobileweb,1158891,1637692,1536149
4,2025-01-02,desktop,472429,774507,953167


In [53]:
cur.execute(c2_by_day_usertype)
dfc2 = cur.fetch_pandas_all()
cur.execute(c2s_by_day_usertype)
dfc2s = cur.fetch_pandas_all()

In [ ]:
# c2_by_day_usertype - PASS
compare_dataframes(dfc2, dfc2s, ['D', 'USER_TYPE'])

DataFrame equality check:
        D  USER_TYPE  USER_COUNT  SESSIONS    PV
0   True       True        True      True  True
1   True       True        True     False  True
2   True       True        True     False  True
3   True       True        True      True  True
4   True       True        True     False  True
5   True       True        True     False  True
6   True       True        True      True  True
7   True       True        True     False  True
8   True       True        True     False  True
9   True       True        True      True  True
10  True       True        True     False  True
11  True       True        True     False  True
12  True       True        True      True  True
13  True       True        True     False  True
14  True       True        True     False  True
15  True       True        True      True  True
16  True       True        True     False  True
17  True       True        True     False  True
18  True       True        True      True  True
19  True     

In [ ]:
dfc2.head() #OK

,D,USER_TYPE,USER_COUNT,SESSIONS,PV
0,2025-01-04,subscriber,4684,11040,21482
1,2025-01-05,anonymous,904862,1329915,1296425
2,2025-01-06,anonymous,2399665,3495975,3272857
3,2025-01-02,registered,45632,95246,182471
4,2025-01-07,subscriber,7592,20464,47359


In [69]:
cur.execute(c3_by_source_usertype)
dfc3 = cur.fetch_pandas_all()
cur.execute(c3s_by_source_usertype)
dfc3s = cur.fetch_pandas_all()

In [70]:
# c3_by_source_usertype
compare_dataframes(dfc3, dfc3s, ['SOURCE', 'USER_TYPE'])

DataFrame equality check:
    SOURCE  USER_TYPE  USER_COUNT  SESSIONS    PV
0    True       True        True      True  True
1    True       True        True     False  True
2    True       True        True     False  True
3    True       True        True      True  True
4    True       True        True     False  True
5    True       True        True     False  True

Session comparison:
                       SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
SOURCE    USER_TYPE                                           
desktop   anonymous        7326072       7326072      0.000000
          registered        588010        586462      0.263261
          subscriber        121933        121872      0.050027
mobileweb anonymous       16174481      16174481      0.000000
          registered        304411        303951      0.151111
          subscriber         52312         52307      0.009558
PASS - difference in session is less than 1%


In [ ]:
dfc3.head() #OK

,SOURCE,USER_TYPE,USER_COUNT,SESSIONS,PV
0,mobileweb,subscriber,4860,52312,102498
1,desktop,anonymous,3513740,7326072,8188188
2,desktop,registered,88977,588010,1231411
3,desktop,subscriber,10974,121933,296666
4,mobileweb,anonymous,9586006,16174481,14603852


In [74]:
cur.execute(c4_by_usertype_referrer)
dfc4 = cur.fetch_pandas_all()
cur.execute(c4s_by_usertype_referrer)
dfc4s = cur.fetch_pandas_all()

In [75]:
# c4_by_usertype_referrer
compare_dataframes(dfc4, dfc4s, ['FIRST_UTM_REFERRER_SIMPLIFIED', 'USER_TYPE'])

DataFrame equality check:
     USER_TYPE  FIRST_UTM_REFERRER_SIMPLIFIED  USER_COUNT  SESSIONS    PV
0        True                           True        True      True  True
1        True                           True        True     False  True
2        True                           True        True      True  True
3        True                           True        True      True  True
4        True                           True        True     False  True
5        True                           True        True     False  True
6        True                           True        True      True  True
7        True                           True        True     False  True
8        True                           True        True     False  True
9        True                           True        True      True  True
10       True                           True        True     False  True
11       True                           True        True     False  True
12       True           

In [ ]:
dfc4.head() #OK

,USER_TYPE,FIRST_UTM_REFERRER_SIMPLIFIED,USER_COUNT,SESSIONS,PV
0,subscriber,Organic,5476,28247,56695
1,subscriber,Other,3199,7221,15113
2,subscriber,Social,469,1339,1900
3,registered,AI agents,2163,4691,8053
4,subscriber,AI agents,159,426,629


In [76]:
cur.execute(c5_by_usertype_country)
dfc5 = cur.fetch_pandas_all()
cur.execute(c5_by_usertype_country)
dfc5s = cur.fetch_pandas_all()

In [77]:
# c5_by_usertype_country
compare_dataframes(dfc5, dfc5s, ['FIRST_COUNTRY', 'USER_TYPE'])

DataFrame equality check:
      USER_TYPE  FIRST_COUNTRY  USER_COUNT  SESSIONS    PV
0         True           True        True      True  True
1         True           True        True      True  True
2         True           True        True      True  True
3         True           True        True      True  True
4         True           True        True      True  True
..         ...            ...         ...       ...   ...
412       True           True        True      True  True
413       True           True        True      True  True
414       True          False        True      True  True
415       True          False        True      True  True
416       True          False        True      True  True

[417 rows x 5 columns]

Session comparison:
                           SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
FIRST_COUNTRY USER_TYPE                                           
Afghanistan   anonymous             74            74           0.0
              registered      

In [ ]:
dfc5.head() # OK

,USER_TYPE,FIRST_COUNTRY,USER_COUNT,SESSIONS,PV
0,registered,France,877,5424,11033
1,registered,Kazakhstan,4,51,79
2,anonymous,Lesotho,10,13,16
3,registered,Chad,1,1,7
4,anonymous,Nauru,2,2,4


In [192]:
dfd1 = run_query(d1_by_day_source_referrer_edition)
dfd1s = run_query(d1s_by_day_source_referrer_edition)

In [193]:
compare_dataframes(dfd1, dfd1s, ['D', 'SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_EDITION'], False)


Session comparison:
                                                                   SESSIONS_df1  \
D          SOURCE    FIRST_UTM_REFERRER_SIMPLIFIED FIRST_EDITION                 
2025-12-01 desktop   AI agents                     jp                       78   
                                                   us                    29370   
                     Direct                        jp                    13572   
                                                   us                   219118   
                     Email                         jp                        1   
...                                                                        ...   
2025-12-03 mobileweb Organic                       us                   320674   
                     Other                         jp                    25612   
                                                   us                   150302   
                     Social                        jp                    299

In [194]:
dfd2 = run_query(d2_by_day_source_referrer_region)
dfd2s = run_query(d2s_by_day_source_referrer_region)

In [195]:
# d2_by_day_source_usertype_referrer_region
d2r = compare_dataframes(dfd2, dfd2s, ['D', 'SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_REGION'], False)


Session comparison:
                                                                  SESSIONS_df1  \
D          SOURCE    FIRST_UTM_REFERRER_SIMPLIFIED FIRST_REGION                 
2025-12-01 desktop   AI agents                     APAC                  2508   
                                                   EMEA                  5796   
                                                   LATAM                  735   
                                                   NA                   17700   
                     Direct                        APAC                  7571   
...                                                                       ...   
2025-12-03 mobileweb Other                         NA                  114827   
                     Social                        APAC                  6829   
                                                   EMEA                  7359   
                                                   LATAM                  329   
      

In [190]:
# e1: date_set, source, first_region 
cur.execute(e1)
dfe1 = cur.fetch_pandas_all()
cur.execute(e1s)
dfe1s = cur.fetch_pandas_all()

In [191]:
dfe1 = derived_metrics(dfe1)
dfe1s = derived_metrics(dfe1s)

In [194]:
compare_dataframes(dfe1, dfe1s, ['DATE_SET', 'FIRST_REGION'], content_flag=False)


Session comparison:
                        SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
DATE_SET FIRST_REGION                                          
set_a    Africa               35779         35779           0.0
         Americas           9190534       9190534           0.0
         Asia                891222        891222           0.0
         Europe             1514649       1514649           0.0
         Oceania             170673        170673           0.0
set_b    Africa               37722         37722           0.0
         Americas           8773888       8773888           0.0
         Asia                874011        874011           0.0
         Europe             1518718       1518718           0.0
         Oceania             203926        203926           0.0
PASS - difference in sessions is less than 1% and dataframes match


'PASS'

In [195]:
# desktop America
dfe1_filtered = dfe1[(dfe1['SOURCE'] == 'desktop') & (dfe1['FIRST_REGION'] == 'Americas')]


In [200]:
# Filter the data
set_a = dfe1_filtered[dfe1_filtered['DATE_SET'] == 'set_a'].reset_index(drop=True)
set_b = dfe1_filtered[dfe1_filtered['DATE_SET'] == 'set_b'].reset_index(drop=True)

change_user = (set_b['USER_COUNT'].fillna(0) - set_a['USER_COUNT'].fillna(0)) / set_a['USER_COUNT'].replace(0, np.nan)
change_pv = (set_b['PV'].fillna(0) - set_a['PV'].fillna(0)) / set_a['PV'].replace(0, np.nan)
change_session = (set_b['SESSIONS'].fillna(0) - set_a['SESSIONS'].fillna(0)) / set_a['SESSIONS'].replace(0, np.nan)

change_pv_s = (set_b['PV/S'].fillna(0) - set_a['PV/S'].fillna(0)) / set_a['PV/S'].replace(0, np.nan)
change_article_recir_perc = (set_b['recirculation_perc'].fillna(0) - set_a['recirculation_perc'].fillna(0))

print("User:", change_user.values, "PV:", change_pv.values, "Sessions:", change_session.values, "PV/S:", change_pv_s.values, "recirc_per:", change_article_recir_perc.values)

User: [-0.08406615] PV: [-0.08738825] Sessions: [-0.09687939] PV/S: [0.01050928] recirc_per: [0.00258573]


In [197]:
set_a

,DATE_SET,SOURCE,FIRST_REGION,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,set_a,desktop,Americas,1623293,3220045,3680598,1666549,530622,83139,1047825,2.267365,1.143027,1.98365,0.284689,0.022588,0.318396


In [198]:
set_b #OK

,DATE_SET,SOURCE,FIRST_REGION,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,set_b,desktop,Americas,1486829,2908089,3358957,1449949,440567,84559,923411,2.259141,1.155039,1.9559,0.27491,0.025174,0.30385


In [107]:
(1486829-1623293) / 1623293

-0.08406615441574626

## Daily Content Review

In [99]:
c1 = run_query(c_a1_overall)
c1s = run_query(c_a1s_overall)

In [100]:
c1

,USER_COUNT,PV
0,3705881,5690382


In [102]:
c1s

,USER_COUNT,PV
0,3705881,5690382


In [ ]:
if (c1 == c1s).all().all():
    result = "Daily Content - C1: Pass"
else:
    result = "Fail"
print(result)

Daily Content - C1: Pass


In [104]:
c2 = run_query(c_a2_by_day)
c2s = run_query(c_a2s_by_day)

In [105]:
# C-A2. content by day - pass
compare_dataframes(c2, c2s, ['D'], True)

'PASS'

In [106]:
c2s

,D,USER_COUNT,PV
0,2025-11-03,1594177,2385006
1,2025-11-01,1237080,1693979
2,2025-11-02,1160253,1611397


In [107]:
c3 = run_query(c_a3_by_topic_ch)
c3s = run_query(c_a3s_by_topic_ch)

In [108]:
# C-A3. c_a3_by_topic_ch
compare_dataframes(c3, c3s, ['TOPIC_CHANNEL'], True)

'PASS'

In [111]:
c3s.head()

,TOPIC_CHANNEL,USER_COUNT,PV
0,رياضة,813,1054
1,press releases,2898,3513
2,cycling,20,20
3,technology,147203,185663
4,static page,1159,1313


In [112]:
c4 = run_query(c_a4_by_usertype)
c4s = run_query(c_a4s_by_usertype)

In [113]:
# C-A4. c_a4_by_usertype 
compare_dataframes(c4, c4s, ['USER_TYPE'], True)

'PASS'

In [114]:
c4.head() #OK

,USER_TYPE,USER_COUNT,PV
0,registered,57627,285114
1,subscriber,22017,174998
2,anonymous,3628749,5230270


In [115]:
c5 = run_query(c_a5_by_country_tch)
c5s = run_query(c_a5s_by_country_tch)

In [116]:
# C-A5. c_a5_by_country_tch
compare_dataframes(c5, c5s, ['FIRST_COUNTRY', 'TOPIC_CHANNEL'], True)

'PASS'

In [117]:
c5.head() #OK

,FIRST_COUNTRY,TOPIC_CHANNEL,USER_COUNT,PV
0,South Africa,home,452,1059
1,United Kingdom of Great Britain and Northern I...,sports,10862,12818
2,Singapore,business,4221,5230
3,United States of America,default,2298,2487
4,Taiwan,science,61,100


In [118]:
c6 = run_query(c_a6_by_region)
c6s = run_query(c_a6s_by_region)

In [119]:
# C-A6. c_a6_by_region
compare_dataframes(c6, c6s, ['FIRST_REGION'], True)

'PASS'

In [120]:
c6.head() #OK

,FIRST_REGION,USER_COUNT,PV
0,None,271228,476754
1,NA,2503274,3683167
2,LATAM,25709,46545
3,EMEA,560114,877649
4,APAC,348218,606267


In [121]:
c7 = run_query(c_a7_by_subchannel)
c7s = run_query(c_a7s_by_subchannel)

In [122]:
compare_dataframes(c7, c7s, ['TOPIC_SUBCHANNEL'], True)

'PASS'

In [123]:
c7.head() #OK

,TOPIC_SUBCHANNEL,USER_COUNT,PV
0,iowa,12,12
1,people moves,17,17
2,stocks,32415,39154
3,lacrosse,17,17
4,رياضة,810,1020


In [124]:
c8 = run_query(c_a8_by_referrer)
c8s = run_query(c_a8s_by_referrer)

In [125]:
compare_dataframes(c8, c8s, ['FIRST_UTM_REFERRER_SIMPLIFIED'], True)

'PASS'

In [126]:
c8.head() #OK

,FIRST_UTM_REFERRER_SIMPLIFIED,USER_COUNT,PV
0,Direct,676110,1552180
1,AI agents,115794,189919
2,Email,5520,6714
3,Other,507007,739870
4,Organic,2148769,2813964


In [127]:
c9 = run_query(c_a9_by_edition)
c9s = run_query(c_a9_by_edition)

In [128]:
compare_dataframes(c9, c9s, ['FIRST_EDITION'], True)

'PASS'

In [129]:
c9.head() #OK

,FIRST_EDITION,USER_COUNT,PV
0,in,1,1
1,None,7094,14579
2,us,3388239,5177464
3,jp,312947,498338


In [131]:
c_b1 = run_query(c_b1_top_canonical_url)
c_b1s = run_query(c_b1s_top_canonical_url)

In [132]:
c_b2 = run_query(c_b2_top_topic_channel_by_source)
c_b2s = run_query(c_b2s_top_topic_channel_by_source)

In [133]:
c_b3 = run_query(c_b3_top_landing_page_user_type)
c_b3s = run_query(c_b3s_top_landing_page_user_type)

In [134]:
c_b4 = run_query(c_b4_top_article)
c_b4s = run_query(c_b4_top_article)

In [135]:
# C-B1. c_b1_top_canonical_url
compare_dataframes(c_b1, c_b1s, ['CANONICAL_URL'], True)

'PASS'

In [136]:
# C-B2. c_b2_top_topic_channel_by_source
compare_dataframes(c_b2, c_b2s, ['FIRST_SOURCE', 'TOPIC_CHANNEL'], True)

'PASS'

In [137]:
# C-B3. c_b3_top_landing_page_user_type
compare_dataframes(c_b3, c_b3s, ['CANONICAL_URL', 'CONTENT_TYPE', 'USER_TYPE'], True)

'PASS'

In [138]:
# C-B4. c_b4_top_article
compare_dataframes(c_b4, c_b4s, ['CANONICAL_URL', 'CONTENT_TYPE'], True)

'PASS'

In [139]:
c_b1 #OK

,CANONICAL_URL,USER_COUNT,PV
0,/home/,360001,762307
1,/world/china/supreme-court-wont-stop-trumps-ta...,231508,246166
2,/legal/litigation/no-jack-o-lantern-no-justice...,179970,187552
3,/world/two-dead-10-injured-shooting-greek-isla...,94473,98768
4,/investigations/narrow-pacific-waterway-is-hea...,89161,97645
5,/world/uk/two-arrested-after-multiple-people-s...,76958,82687
6,/world/,32863,67859
7,/world/china/trump-says-christians-face-existe...,60375,67196
8,/world/uk/uk-counterterrorism-police-investiga...,59621,64367
9,/world/india/fearing-fraud-canada-rejects-most...,37780,40019


In [140]:
c_b2 #OK

,FIRST_SOURCE,TOPIC_CHANNEL,USER_COUNT,PV
0,desktop,world,356206,564568
1,desktop,home,190135,372521
2,desktop,business,213269,312138
3,desktop,markets,147607,229409
4,desktop,technology,66305,92955
5,desktop,None,41153,80800
6,desktop,sustainability,52461,70267
7,desktop,ホーム,20304,46076
8,desktop,legal,33642,42544
9,desktop,マーケット,24828,40710


In [141]:
c_b3

,USER_TYPE,CONTENT_TYPE,CANONICAL_URL,USER_COUNT,PV
0,subscriber,landing page,/home/,12499,55007
1,subscriber,landing page,/world/,1532,5185
2,subscriber,landing page,/business/,908,2323
3,subscriber,landing page,/world/us/,809,2246
4,subscriber,landing page,/markets/,802,1667
5,subscriber,landing page,/world/china/,429,1366
6,subscriber,landing page,/world/europe/,430,1011
7,subscriber,landing page,/business/energy/,346,875
8,subscriber,landing page,/world/americas/,318,658
9,subscriber,landing page,/technology/,264,645


In [142]:
c_b4

,CONTENT_TYPE,CANONICAL_URL,USER_COUNT,PV
0,article,/world/china/supreme-court-wont-stop-trumps-ta...,231508,246166
1,article,/world/two-dead-10-injured-shooting-greek-isla...,94473,98768
2,article,/investigations/narrow-pacific-waterway-is-hea...,89161,97645
3,article,/world/uk/two-arrested-after-multiple-people-s...,76958,82687
4,article,/world/china/trump-says-christians-face-existe...,60375,67196
5,article,/world/uk/uk-counterterrorism-police-investiga...,59621,64367
6,article,/world/india/fearing-fraud-canada-rejects-most...,37780,40019
7,article,/world/europe/ukraine-lands-special-forces-emb...,32555,35081
8,article,/business/aerospace-defense/russia-uses-missil...,29510,31856
9,article,/business/healthcare-pharmaceuticals/kimberly-...,27340,31706


# MONTHLY KPI

## QUERY

In [144]:
from datetime import datetime

# Input string date
start_date = '2025-11-01'
end_date = '2025-11-30'
date_str = start_date

# Convert to datetime object
date_obj = datetime.strptime(date_str, '%Y-%m-%d')

# Truncate to month (first day of the month)
month_truncated = date_obj.replace(day=1)

# Output
month = month_truncated.strftime('%Y-%m-%d')
print(month)

2025-11-01


In [78]:
# M-A1. Overall 
ma0 = f"""select date_trunc('month', e.d) as month,
    count(distinct coalesce(user_dim_id, ss_anonymous_user_id)) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1"""

ma0s = f"""select month, 
    user_count,
    sessions,
    pageviews,
    VIDEO_STORY_START,
    VIDEO_STORY_FINISHED,
    ARTICLE_RECIRCULATION,
    ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month between '{start_date}' and '{end_date}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL'"""

ma1 = f"""select
    count(distinct coalesce(user_dim_id, ss_anonymous_user_id)) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' """

ma1s = f"""select
    user_count,
    sessions,
    pageviews,
    VIDEO_STORY_START,
    VIDEO_STORY_FINISHED,
    ARTICLE_RECIRCULATION,
    ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL'"""

# M-A2. Metrics by source
ma2 = f"""select
    source as first_source,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1;"""

ma2s = f"""select first_source,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source != 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL' 
    group by 1;"""

# M-A3. Metrics by edition
ma3 = f"""select
    ed.edition as first_edition,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1;"""

ma3s = f"""select first_edition,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition != 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL' 
    group by 1;"""

# M-A4. Metrics by referrer
ma4 = f"""select
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1;"""

ma4s = f"""select first_utm_referrer_simplified,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified != 'ALL'
    and first_country = 'ALL' and first_region = 'ALL' 
    group by 1;"""

# M-A5. Metrics by country
ma5 = f"""select
    c.country_name as first_country,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1;"""

ma5s = f"""select first_country,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country != 'ALL' and first_region = 'ALL' 
    group by 1;"""

# M-A6. Metrics by region
ma6 = f"""select
    c.region as first_region,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1;"""

ma6s = f"""select first_region,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region != 'ALL' 
    group by 1;"""

# M-A7. Metrics by user type
ma7 = f"""select
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1;"""

ma7s = f"""select user_type,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type != 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL' 
    group by 1;"""

# M-A8. Metrics by source & edition
ma8 = f"""select
    s.source as first_source,
    ed.edition as first_edition,
    count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2;"""

ma8s = f"""select first_source, first_edition,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source != 'ALL' and first_edition != 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL' 
    group by 1,2;"""


# M-A9. Metrics by referrer & country
ma9 = f"""select
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED,
    c.country_name as first_country,
    count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2;"""

ma9s = f"""select first_utm_referrer_simplified, first_country,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified != 'ALL'
    and first_country != 'ALL' and first_region = 'ALL' 
    group by 1,2;"""

# M-A10. Metrics by region & user type
ma10 = f"""select
    c.region_name as first_region,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2;"""

ma10s = f"""select first_region, user_type,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type != 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region != 'ALL' 
    group by 1,2;"""

# M-A11. Metrics by source, edition, referrer
ma11 = f"""select s.source as first_source, ed.edition as first_edition,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1,2,3;"""

ma11s = f"""select first_source, first_edition, first_utm_referrer_simplified, 
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source != 'ALL' and first_edition != 'ALL' and first_utm_referrer_simplified != 'ALL'
    and first_country = 'ALL' and first_region = 'ALL' 
    group by 1,2,3;"""

# M-A12. Metrics by country, region, user type
ma12 = f"""select c.country_name as first_country,
    c.region_name as first_region,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2,3;"""

ma12s = f"""select first_country, first_region, user_type,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type != 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country != 'ALL' and first_region != 'ALL' 
    group by 1,2,3;"""

# M-A13. Metrics by source, region, referrer, user type
ma13 = f"""select
    s.source as first_source,
    c.region_name as first_region,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2,3,4;"""

ma13s = f"""select first_source, first_region, first_utm_referrer_simplified, user_type,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type != 'ALL' and first_source != 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified != 'ALL'
    and first_country = 'ALL' and first_region != 'ALL' 
    group by 1,2,3,4;"""

# M-A14. Metrics by source, edition, region, referrer, user type, country
ma14 = f"""select
    s.source as first_source,
    ed.edition as first_edition,
    c.region as first_region,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    c.country_name as first_country,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and segment_user_agent_type = 'other user agent' group by 1,2,3,4,5,6;"""

ma14s = f"""select first_source, first_edition, first_region, first_utm_referrer_simplified, user_type, first_country,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type != 'ALL' and first_source != 'ALL' and first_edition != 'ALL' and first_utm_referrer_simplified != 'ALL'
    and first_country != 'ALL' and first_region != 'ALL' 
    group by 1,2,3,4,5,6;"""


# M-A15. Metrics by month & user type
ma15 = f"""select
    date_trunc('month', e.d) as month,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct e.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' group by 1,2;"""

ma15s = f"""select month, user_type,
    sum(user_count) as user_count,
    sum(sessions) as sessions,
    sum(pageviews) as pageviews,
    sum(VIDEO_STORY_START) as video_story_start,
    sum(VIDEO_STORY_FINISHED) as video_story_finished,
    sum(ARTICLE_RECIRCULATION) as article_recirculation,
    sum(ARTICLE_BOTTOM_REACH) as article_bottom_reach 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_KPI_SUMMARY
where month = '{month}' 
    and user_type != 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL' 
    group by 1,2;"""

In [149]:
# MC-A0. Overall 
mca0 = f"""select date_trunc('month', e.d) as month, TOPIC_CHANNEL,
    count(distinct coalesce(e.user_dim_id, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
where e.d between '{start_date}' and '{end_date}' AND TOPIC_CHANNEL IS NOT NULL and segment_user_agent_type = 'other user agent'
    group by 1,2;"""

mca0s = f"""select month, TOPIC_CHANNEL, user_count, pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL'
    ;"""

# MC-A1. Metrics by month, source, edition, region, referrer, user type, country, topic channel
mca1 = f"""select date_trunc('month', e.d) as month, e.TOPIC_CHANNEL,
    s.source as first_source,
    ed.edition as first_edition,
    c.region as first_region,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    c.country_name as first_country,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}'  and segment_user_agent_type = 'other user agent'
    AND TOPIC_CHANNEL IS NOT NULL 
    group by 1,2,3,4,5,6,7,8;"""

mca1s = f"""select month, topic_channel, first_source, first_edition, first_region, first_utm_referrer_simplified, user_type, first_country,
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type != 'ALL' and first_source != 'ALL' and first_edition != 'ALL' and first_utm_referrer_simplified != 'ALL'
    and first_country != 'ALL' and first_region != 'ALL'
    group by 1,2,3,4,5,6,7,8;"""

# MC-A2. Metrics by user type
mca2 = f"""select date_trunc('month', e.d) as month, topic_channel,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
where e.d between '{start_date}' and '{end_date}'  and segment_user_agent_type = 'other user agent'
    AND TOPIC_CHANNEL IS NOT NULL 
    group by 1,2,3;"""

mca2s = f"""SELECT month, topic_channel, user_type,
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month between '{start_date}' and '{end_date}' 
    and user_type != 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL'
    group by 1,2,3;"""

# MC-A3. Metrics by source
mca3 = f"""select date_trunc('month', e.d) as month, topic_channel,
    s.source as first_source,
    count(distinct coalesce(e.user_dim_id, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    AND TOPIC_CHANNEL IS NOT NULL and segment_user_agent_type = 'other user agent'
    group by 1,2,3;"""

mca3s = f"""SELECT month, topic_channel, first_source,
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source != 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL'
    group by 1,2,3;"""

# MC-A4. Metrics by edition
mca4 = f"""select date_trunc('month', e.d) as month, topic_channel,
    ed.edition as first_edition,
    count(distinct coalesce(e.user_dim_id, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
where e.d between '{start_date}' and '{end_date}' 
    AND TOPIC_CHANNEL IS NOT NULL and segment_user_agent_type = 'other user agent'
    group by 1,2,3;"""

mca4s = f"""SELECT month, topic_channel, first_edition,
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition != 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region = 'ALL'
    group by 1,2,3;"""

# MC-A5. Metrics by referrer
mca5 = f"""select date_trunc('month', e.d) as month, e.TOPIC_CHANNEL,
    CASE WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'direct - direct' THEN 'Direct' 
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) = 'email' THEN 'Email'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'ai agents - %' THEN 'AI agents'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'newsletter - %' THEN 'Newsletter'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'organic - %' THEN 'Organic'
        WHEN lower(trim(S.FIRST_UTM_REFERRER_NULL_ALLOWED)) LIKE 'social - %' THEN 'Social'
        ELSE 'Other' END AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct coalesce(e.user_dim_id, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
where e.d between '{start_date}' and '{end_date}' 
    AND TOPIC_CHANNEL IS NOT NULL and segment_user_agent_type = 'other user agent'
    group by 1,2,3;"""

mca5s = f"""select month, topic_channel, first_utm_referrer_simplified,
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified != 'ALL'
    and first_country = 'ALL' and first_region = 'ALL'
    group by 1,2,3;"""

# MC-A6. Metrics by country
mca6 = f"""select date_trunc('month', e.d) as month, e.TOPIC_CHANNEL,
    c.country_name as first_country,
    count(distinct coalesce(e.user_dim_id, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' 
    AND TOPIC_CHANNEL IS NOT NULL and segment_user_agent_type = 'other user agent'
    group by 1,2,3;"""

mca6s = f"""select month, topic_channel, first_country,
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country != 'ALL' and first_region = 'ALL'
    group by 1,2,3;"""

# MC-A7. Metrics by region
mca7 = f"""select date_trunc('month', e.d) as month, e.TOPIC_CHANNEL,
    c.region as first_region,
    count(distinct coalesce(e.user_dim_id, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' 
    AND TOPIC_CHANNEL IS NOT NULL and segment_user_agent_type = 'other user agent'
    group by 1,2,3;"""

mca7s = f"""select month, topic_channel, first_region, 
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source = 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region != 'ALL'
    group by 1,2,3;"""

# MC-A8. Metrics by region, SOURCE
mca8 = f"""select date_trunc('month', e.d) as month, e.TOPIC_CHANNEL,
    s.source as first_source,
    c.region as first_region,
    count(distinct coalesce(e.user_dim_id, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct iff(event_type = 'page', e.ss_event_id, null)) as pageviews
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' 
    AND TOPIC_CHANNEL IS NOT NULL and segment_user_agent_type = 'other user agent'
    group by 1,2,3,4;"""

mca8s = f"""select month, topic_channel, first_source, first_region,
    sum(user_count) as user_count,
    sum(pageviews) as pageviews
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_MONTHLY_TOPIC_SUMMARY
where month = '{month}' 
    and user_type = 'ALL' and first_source != 'ALL' and first_edition = 'ALL' and first_utm_referrer_simplified = 'ALL'
    and first_country = 'ALL' and first_region != 'ALL'
    group by 1,2,3,4;"""

## Monthly KPI Review

In [8]:
df_ma0 = run_query(ma0)
df_ma0s = run_query(ma0s)

In [12]:
df_ma0

,MONTH,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,2025-11-01,32816794,67696773,65626024,36113295,2744070,1875520,19105065


In [13]:
df_ma0s

,MONTH,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,2025-11-01,32816794,67697015,65626024,36113295,2744070,1875520,19105065


In [11]:
compare_dataframes(df_ma0, df_ma0s, ['MONTH'], 0)


Session comparison:
             SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
MONTH                                               
2025-11-01      67696773      67697015     -0.000357
FAIL - either session difference exceeds 1% or dataframes do not match


In [180]:
df_ma15 = run_query(ma15)
df_ma15s = run_query(ma15s)

In [182]:
df_ma15

,MONTH,USER_TYPE,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,2025-10-01,subscriber,49845,1158982,2469259,230978,107284,64225,439358
1,2025-08-01,subscriber,43442,987859,2063621,194239,89233,53028,362238
2,2025-09-01,registered,236504,2229452,4079766,2152490,205457,77867,699765
3,2025-09-01,anonymous,39162934,82121528,93772214,42113032,3997122,1516749,23523398
4,2025-10-01,anonymous,38086681,78773197,94074977,40443877,3190202,1499272,22389716
5,2025-10-01,registered,230275,2200788,4027790,2118002,194667,81569,706902
6,2025-08-01,registered,180437,2015941,3729769,1809091,294934,73872,625526
7,2025-09-01,subscriber,47743,1096645,2377044,245868,112617,65716,444390
8,2025-08-01,anonymous,30981454,67075475,75129347,29591834,3848519,1287428,18632290


In [183]:
df_ma15s

,MONTH,USER_TYPE,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,2025-08-01,subscriber,43442,987563,2063621,194239,89233,53028,362238
1,2025-08-01,registered,180437,2015925,3729769,1809091,294934,73872,625526
2,2025-08-01,anonymous,30981454,67075475,75129347,29591834,3848519,1287428,18632290


In [181]:
compare_monthly_dataframes(df_ma15, df_ma15s, ['MONTH', 'USER_TYPE'], 0)

ValueError: Can only compare identically-labeled (both index and columns) DataFrame objects

In [14]:
df_ma1 = run_query(ma1)
df_ma1s = run_query(ma1s)

In [15]:
df_ma1s

,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,32816794,67697015,65626024,36113295,2744070,1875520,19105065


In [17]:
df_ma1

,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,32816794,67696773,65626024,36113295,2744070,1875520,19105065


In [16]:
df_ma1 == df_ma1s

,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,True,False,True,True,True,True,True


In [44]:
df_ma2 = run_query(ma2)
df_ma2s = run_query(ma2s)

In [45]:
df_ma2

,FIRST_SOURCE,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,desktop,9345563,22952078,25354472,16023771,1858176,843284,6003428
1,mobileweb,23488861,44744695,40271552,20089524,885894,1032236,13101637


In [46]:
df_ma2s

,FIRST_SOURCE,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,desktop,9345563,22952266,25354472,16023771,1858176,843284,6003428
1,mobileweb,23488861,44744749,40271552,20089524,885894,1032236,13101637


In [48]:
compare_dataframes(df_ma2, df_ma2s, ['FIRST_SOURCE'], False)


Session comparison:
               SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
FIRST_SOURCE                                          
desktop           22952078      22952266     -0.000819
mobileweb         44744695      44744749     -0.000121
PASS - difference in sessions is less than 1% and dataframes match


In [49]:
df_ma3 = run_query(ma3)
df_ma3s = run_query(ma3s)

In [56]:
df_ma4 = run_query(ma4)
df_ma4s = run_query(ma4s)

In [58]:
df_ma5 = run_query(ma5)
df_ma5s = run_query(ma5s)

In [88]:
df_ma6 = run_query(ma6)
df_ma6s = run_query(ma6s)

In [71]:
df_ma7 = run_query(ma7)
df_ma7s = run_query(ma7s)

In [54]:
compare_monthly_dataframes(df_ma3, df_ma3s, ['FIRST_EDITION'], 1)


Session comparison:
                SESSIONS_df1  SESSIONS_df2   DIFF  %_DIFFERENCE
FIRST_EDITION                                                 
                   113236.0      113236.0    0.0      0.000000
br                      1.0           1.0    0.0      0.000000
cn                      4.0           4.0    0.0      0.000000
fr                      1.0           1.0    0.0      0.000000
in                      7.0           7.0    0.0      0.000000
it                      1.0           1.0    0.0      0.000000
jp                8686240.0     8686240.0    0.0      0.000000
ru                      2.0           2.0    0.0      0.000000
uk                     17.0          17.0    0.0      0.000000
us               58897264.0    58897506.0  242.0     -0.000411


'PASS'

In [57]:
compare_monthly_dataframes(df_ma4, df_ma4s, ['FIRST_UTM_REFERRER_SIMPLIFIED'], 1)


Session comparison:
                                SESSIONS_df1  SESSIONS_df2   DIFF  %_DIFFERENCE
FIRST_UTM_REFERRER_SIMPLIFIED                                                 
AI agents                         1938586.0     1938589.0    3.0     -0.000155
Direct                           14620279.0    14620411.0  132.0     -0.000903
Email                              117432.0      117432.0    0.0      0.000000
Newsletter                         373360.0      373364.0    4.0     -0.001071
Organic                          36190908.0    36190987.0   79.0     -0.000218
Other                            10747489.0    10747512.0   23.0     -0.000214
Social                            3708719.0     3708720.0    1.0     -0.000027


'PASS'

In [59]:
compare_monthly_dataframes(df_ma5, df_ma5s, ['FIRST_COUNTRY'], 0)


Session comparison:
                            SESSIONS_df1  SESSIONS_df2  DIFF  %_DIFFERENCE
FIRST_COUNTRY                                                            
                              7189862.0     7189874.0  12.0     -0.000167
Afghanistan                       288.0         288.0   0.0      0.000000
Aland Islands                      48.0          48.0   0.0      0.000000
Albania                           750.0         750.0   0.0      0.000000
Algeria                          2068.0        2068.0   0.0      0.000000
...                                 ...           ...   ...           ...
Wallis and Futuna Islands           4.0           4.0   0.0      0.000000
Western Sahara                      3.0           3.0   0.0      0.000000
Yemen                            1517.0        1517.0   0.0      0.000000
Zambia                           1479.0        1479.0   0.0      0.000000
Zimbabwe                         1918.0        1918.0   0.0      0.000000

[240 rows x 4 c

'PASS'

In [89]:
compare_monthly_dataframes(df_ma6, df_ma6s, ['FIRST_REGION'], 0)


Session comparison:
               SESSIONS_df1  SESSIONS_df2   DIFF  %_DIFFERENCE
FIRST_REGION                                                 
                 7189862.0     7189874.0   12.0     -0.000167
APAC             6878623.0     6878650.0   27.0     -0.000393
EMEA             9197002.0     9197041.0   39.0     -0.000424
LATAM             569186.0      569187.0    1.0     -0.000176
NA              43862100.0    43862263.0  163.0     -0.000372


'PASS'

In [73]:
compare_monthly_dataframes(df_ma7, df_ma7s, ['USER_TYPE'], 0)


Session comparison:
             SESSIONS_df1  SESSIONS_df2   DIFF  %_DIFFERENCE
USER_TYPE                                                  
anonymous     65058475.0    65058980.0  505.0     -0.000776
registered     1769869.0     1769860.0    9.0      0.000509
subscriber      993691.0      993291.0  400.0      0.040254


'PASS'

In [114]:
df_ma8 = run_query(ma8)
df_ma8s = run_query(ma8s)

In [115]:
df_ma8.head()

,FIRST_SOURCE,FIRST_EDITION,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,mobileweb,us,3884610,6152201,5602303,3133033,389357,97978,2218247
1,desktop,us,1248188,2473313,2981649,1314542,432478,70424,805906
2,mobileweb,None,48,50,1,0,0,0,0
3,mobileweb,jp,473097,742198,749707,15973,3160,40428,333234
4,desktop,in,2,2,3,0,0,1,0


In [117]:
df_ma8s.head()

,FIRST_SOURCE,FIRST_EDITION,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,mobileweb,,48,50,1,0,0,0,0
1,mobileweb,us,3884610,6152201,5602303,3133033,389357,97978,2218247
2,mobileweb,jp,473097,742198,749707,15973,3160,40428,333234
3,desktop,in,2,2,3,0,0,1,0
4,desktop,,217,276,79,52,21,0,43


In [119]:
compare_monthly_dataframes(df_ma8, df_ma8s, ['FIRST_EDITION', 'FIRST_SOURCE'])

'PASS'

In [120]:
df_ma9 = run_query(ma9)
df_ma9s = run_query(ma9s)
df_ma10 = run_query(ma10)
df_ma10s = run_query(ma10s)

In [123]:
compare_monthly_dataframes(df_ma9, df_ma9s, ['FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_COUNTRY'])

'PASS'

In [130]:
df_ma10

,FIRST_REGION,USER_TYPE,USER_COUNT,SESSIONS,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,Americas,registered,58821,221073,359956,142362,47338,7836,63869
1,None,registered,4962,21337,47475,21584,9643,1406,10169
2,Africa,registered,370,1358,3081,1180,254,77,698
3,Americas,anonymous,4096903,6697244,6358594,3309681,547421,119890,2289424
4,Americas,subscriber,9285,58560,125403,58724,15674,4104,28502
5,Africa,subscriber,2,10,18,18,10,0,4
6,Oceania,registered,1439,5316,7458,2580,667,126,1280
7,Africa,anonymous,20971,26631,27457,16073,2823,544,13168
8,Oceania,subscriber,148,1056,2428,1157,281,73,725
9,Oceania,anonymous,96025,146743,136087,80803,11221,3156,66692


In [132]:
compare_monthly_dataframes(df_ma10, df_ma10s, ['USER_TYPE', 'FIRST_REGION'])


Session comparison:
                          SESSIONS_df1  SESSIONS_df2  %_DIFFERENCE
USER_TYPE  FIRST_REGION                                          
anonymous                    833255.0      833255.0      0.000000
           Africa             26631.0       26631.0      0.000000
           Americas         6697244.0     6697244.0      0.000000
           Asia              581194.0      581194.0      0.000000
           Europe            966473.0      966473.0      0.000000
           Oceania           146743.0      146743.0      0.000000
registered                    21337.0       21314.0      0.107794
           Africa              1358.0        1358.0      0.000000
           Americas          221073.0      220435.0      0.288592
           Asia               38302.0       38255.0      0.122709
           Europe             64176.0       64089.0      0.135565
           Oceania             5316.0        5306.0      0.188111
subscriber                     1206.0        1204.0   

'PASS'

In [79]:
df_ma11 = run_query(ma11)
df_ma11s = run_query(ma11s)
#df_ma12 = run_query(ma12)
#df_ma12s = run_query(ma12s)

In [80]:
#df_ma13 = run_query(ma13)
#df_ma13s = run_query(ma13s)
df_ma14 = run_query(ma14)
df_ma14s = run_query(ma14s)

In [81]:
# M-A11. Metrics by source, edition, referrer
compare_monthly_dataframes(df_ma11, df_ma11s, ['FIRST_SOURCE', 'FIRST_EDITION', 'FIRST_UTM_REFERRER_SIMPLIFIED'], 0)


Session comparison:
                                                           SESSIONS_df1  \
FIRST_SOURCE FIRST_EDITION FIRST_UTM_REFERRER_SIMPLIFIED                 
desktop                    AI agents                             732.0   
                           Direct                              27949.0   
                           Newsletter                            107.0   
                           Organic                             23057.0   
                           Other                                3547.0   
                           Social                                706.0   
             br            Direct                                  1.0   
             cn            Direct                                  2.0   
                           Other                                   2.0   
             fr            Other                                   1.0   
             in            Direct                                  4.0   
                

'PASS'

In [169]:
#M-A12. Metrics by country, region, user type
compare_monthly_dataframes(df_ma12, df_ma12s, ['FIRST_COUNTRY', 'FIRST_REGION', 'USER_TYPE'])


Session comparison:
                                        SESSIONS_df1  SESSIONS_df2  DIFF  \
FIRST_COUNTRY FIRST_REGION USER_TYPE                                      
                           anonymous       815110.0      815110.0   0.0   
                           registered       17170.0       17152.0  18.0   
                           subscriber        1133.0        1131.0   2.0   
Afghanistan   Asia         anonymous           32.0          32.0   0.0   
Aland Islands Europe       anonymous           11.0          11.0   0.0   
...                                             ...           ...   ...   
Yemen         Asia         registered          28.0          28.0   0.0   
Zambia        Africa       anonymous          108.0         108.0   0.0   
                           registered          13.0          13.0   0.0   
Zimbabwe      Africa       anonymous          138.0         138.0   0.0   
                           registered          32.0          32.0   0.0   

  

'PASS'

In [165]:
#M-A13. Metrics by source, region, referrer, user type
compare_monthly_dataframes(df_ma13, df_ma13s, ['FIRST_SOURCE', 'FIRST_REGION', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'USER_TYPE'])


Session comparison:
                                                                     SESSIONS_df1  \
FIRST_SOURCE FIRST_REGION FIRST_UTM_REFERRER_SIMPLIFIED USER_TYPE                  
desktop                   AI agents                     anonymous         7918.0   
                                                        registered         109.0   
                                                        subscriber           5.0   
                          Direct                        anonymous       100841.0   
                                                        registered       12579.0   
...                                                                          ...   
mobileweb    Oceania      Other                         registered          94.0   
                                                        subscriber          11.0   
                          Social                        anonymous         7390.0   
                                                      

'PASS'

In [87]:
#M-A14. Metrics by source, edition, region, referrer, user type, country
compare_monthly_dataframes(df_ma14, df_ma14s, ['FIRST_SOURCE', 'FIRST_EDITION', 'FIRST_REGION', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'USER_TYPE', 'FIRST_COUNTRY'], 200)


Session comparison:
                                                                                                            SESSIONS_df1  \
FIRST_SOURCE FIRST_EDITION FIRST_REGION FIRST_UTM_REFERRER_SIMPLIFIED USER_TYPE  FIRST_COUNTRY                            
desktop                                 AI agents                     anonymous                                   138.0   
                                                                      registered                                   12.0   
                                                                      subscriber                                    2.0   
                                        Direct                        anonymous                                  2597.0   
                                                                      registered                                  340.0   
...                                                                                                                 .

'PASS'

## Monthly Content Review

In [146]:
df_mca0 = run_query(mca0)
df_mca0s = run_query(mca0s)

In [147]:
compare_monthly_content(df_mca0, df_mca0s, ['MONTH', 'TOPIC_CHANNEL'], 1)

'PASS'

In [148]:
df_mca0s.sort_values(by='PAGEVIEWS', ascending=False).head(10)

,MONTH,TOPIC_CHANNEL,USER_COUNT,PAGEVIEWS
34,2025-11-01,world,12842288,20049454
64,2025-11-01,business,7453579,10526782
119,2025-11-01,home,2082246,7438876
5,2025-11-01,markets,4091416,6355409
103,2025-11-01,マーケット,1637052,3105509
24,2025-11-01,ワールド,1640589,2790692
77,2025-11-01,legal,2147683,2590307
66,2025-11-01,sustainability,1924118,2533003
128,2025-11-01,technology,1469767,2050458
36,2025-11-01,ホーム,233374,1136070


In [150]:
df_mca1 = run_query(mca1)
df_mca1s = run_query(mca1s)

In [151]:
df_mca1.head()

,MONTH,TOPIC_CHANNEL,FIRST_SOURCE,FIRST_EDITION,FIRST_REGION,FIRST_UTM_REFERRER_SIMPLIFIED,USER_TYPE,FIRST_COUNTRY,USER_COUNT,PAGEVIEWS
0,2025-11-01,news assistant,mobileweb,us,EMEA,Social,anonymous,Comoros,1,0
1,2025-11-01,default,mobileweb,us,EMEA,Direct,subscriber,United Kingdom of Great Britain and Northern I...,1,0
2,2025-11-01,news assistant,desktop,us,EMEA,Other,anonymous,Slovenia,5,0
3,2025-11-01,news assistant,mobileweb,us,EMEA,AI agents,anonymous,Kyrgyzstan,6,0
4,2025-11-01,news assistant,mobileweb,jp,APAC,Social,anonymous,Japan,965,0


In [152]:
compare_monthly_content(df_mca1, df_mca1s, ['MONTH', 'TOPIC_CHANNEL', 'FIRST_SOURCE', 'FIRST_EDITION', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_COUNTRY', 'FIRST_REGION', 'USER_TYPE'], 1)

'PASS'

In [153]:
df_mca2 = run_query(mca2)
df_mca2s = run_query(mca2s)

In [154]:
compare_monthly_content(df_mca2, df_mca2s, ['MONTH', 'TOPIC_CHANNEL', 'USER_TYPE'], 1)

'PASS'

In [156]:
df_mca2[df_mca2['USER_TYPE'] == 'anonymous']

,MONTH,TOPIC_CHANNEL,USER_TYPE,USER_COUNT,PAGEVIEWS
0,2025-11-01,ビジネス,anonymous,327812,430948
2,2025-11-01,news - topgeneralnews,anonymous,1,1
3,2025-11-01,default,anonymous,47685,55828
4,2025-11-01,blogs,anonymous,1,0
5,2025-11-01,english,anonymous,13686,16344
...,...,...,...,...,...
250,2025-11-01,azienda,anonymous,2259,4794
258,2025-11-01,investigations,anonymous,8527,10694
262,2025-11-01,static,anonymous,664,761
268,2025-11-01,world,anonymous,12689621,18486048


In [157]:
df_mca3 = run_query(mca3)
df_mca3s = run_query(mca3s)

In [158]:
compare_monthly_content(df_mca3, df_mca3s, ['MONTH', 'TOPIC_CHANNEL', 'FIRST_SOURCE'], 1)

'PASS'

In [159]:
df_mca4 = run_query(mca4)
df_mca4s = run_query(mca4s)

In [160]:
compare_monthly_content(df_mca4, df_mca4s, ['MONTH', 'TOPIC_CHANNEL', 'FIRST_EDITION'], 1)

'PASS'

In [161]:
df_mca5 = run_query(mca5)
df_mca5s = run_query(mca5s)

In [162]:
compare_monthly_content(df_mca5, df_mca5s, ['MONTH', 'TOPIC_CHANNEL', 'FIRST_UTM_REFERRER_SIMPLIFIED'], 1)

'PASS'

In [163]:
df_mca6 = run_query(mca6)
df_mca6s = run_query(mca6s)

In [164]:
compare_monthly_content(df_mca6, df_mca6s, ['MONTH', 'TOPIC_CHANNEL', 'FIRST_COUNTRY'], 1)

'PASS'

In [165]:
df_mca7 = run_query(mca7)
df_mca7s = run_query(mca7s)

In [166]:
compare_monthly_content(df_mca7, df_mca7s, ['MONTH', 'TOPIC_CHANNEL', 'FIRST_REGION'], 1)

'PASS'

In [167]:
df_mca8 = run_query(mca8)
df_mca8s = run_query(mca8s)

In [168]:
compare_monthly_content(df_mca8, df_mca8s, ['MONTH', 'TOPIC_CHANNEL', 'FIRST_REGION', 'FIRST_SOURCE'], 1)

'PASS'

## Documentation for Exceptional Cases

- Jan 1–31, 2025 data was used for review.
- The numbers in Tableau (`TOPLINE_SILVER_V1` file and the summary tables) are aligned with the figures pulled from `SILVER_EVENT_DIM`.
- One exception is the **session count**. Theoretically, `USER_DIM_ID` should not be null when `application_authenticated_state` is either "registered" or "subscriber", and the query is built based on that assumption. However, in rare cases, `USER_DIM_ID` is recorded as null even when `application_authenticated_state` is "registered" or "subscriber".
- Due to this technical glitch, the number of sessions by user type may differ slightly for "subscriber" and "registered" users. The discrepancy is less than 1%, and the review considers the numbers aligned if the difference in session count is below 1% for these user types.  

In [ ]:
# SESSION COUNTS
query1 = """
WITH T1 AS (select D,
    count(distinct ss_session_id) as sessions,
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where d between '2026-01-01' and '2026-01-05' 
        and segment_user_agent_type='other user agent' 
        AND SOURCE IN ('desktop', 'mobileweb') GROUP BY 1),
    T2 AS (SELECT D, 
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
where d between '2026-01-01' and '2026-01-05' AND SOURCE IN ('desktop', 'mobileweb') GROUP BY 1)
SELECT T1.D, T1.SESSIONS, T2.SESSIONS AS DKS_SESSIONS
FROM T1 JOIN T2 ON T1.D = T2.D;"""

The sessions in KPI SUMMARY is +8, which is 8/1959332 (0.00000408302). 